In [14]:
import warnings
warnings.filterwarnings('ignore')
import hoomd
import datetime
import gsd
import matplotlib.pyplot as plt
import numpy as np
import gsd.hoomd
from flowermd.base import Pack,Lattice, Simulation
from flowermd.library import EllipsoidForcefield, EllipsoidChain
from flowermd.utils import get_target_box_number_density
from flowermd.utils.constraints import create_rigid_ellipsoid_chain
import unyt as u
import hoomd

In [32]:
LPAR = 1.0
LPERP = 0.5

ellipsoid_chain = EllipsoidChain(lengths=1,num_mols=100,lpar=LPAR,bead_mass=1.0)
ff = EllipsoidForcefield(epsilon=1.0,lpar=LPAR,lperp=LPERP,r_cut=2.0)
ff.hoomd_forces
system = Pack(molecules=ellipsoid_chain, density=0.01*u.Unit("nm**-3"), packing_expand_factor=6,edge=2,overlap=1,fix_orientation=True)

time_string = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
GSD_FILE_PATH = 'logs/ellipsoid' + time_string + '.gsd'
LOG_FILE_PATH = 'logs/ellipsoid' + time_string + '.txt'

rigid_frame, rigid = create_rigid_ellipsoid_chain(
    system.hoomd_snapshot
)
ellipsoid_sim = Simulation(
    initial_state=rigid_frame,
    forcefield=ff.hoomd_forces,
    constraint=rigid,
    gsd_write_freq=int(8e3),
    gsd_file_name=gsd_path,
    log_write_freq=int(1e4),
    log_file_name=LOG_FILE_PATH,
    dt=0.0001
)

target_box = get_target_box_number_density(density=0.9*u.Unit("nm**-3"), n_beads=100)
ellipsoid_sim.run_update_volume(final_box_lengths=target_box, kT=7.0, n_steps=1e6, tau_kt=5*ellipsoid_sim.dt, period=10, thermalize_particles=True)
print("shrink finished")
#ellipsoid_sim.run_NVT(n_steps=5e4, kT=1.0, tau_kt=5*ellipsoid_sim.dt)
print("simulation finished")
#ellipsoid_sim.save_restart_gsd("restart.gsd")
ellipsoid_sim.flush_writers()
#ellipsoid_sim.save_simulation("sim.pickle")

Initializing simulation state from a gsd.hoomd.Frame.
Step 9000 of 1000000; TPS: 5552.58; ETA: 3.0 minutes
Step 18000 of 1000000; TPS: 7450.13; ETA: 2.2 minutes
Step 27000 of 1000000; TPS: 7919.22; ETA: 2.0 minutes
Step 36000 of 1000000; TPS: 7884.9; ETA: 2.0 minutes
Step 45000 of 1000000; TPS: 8154.37; ETA: 2.0 minutes
Step 54000 of 1000000; TPS: 8371.37; ETA: 1.9 minutes
Step 63000 of 1000000; TPS: 8720.68; ETA: 1.8 minutes
Step 72000 of 1000000; TPS: 8875.27; ETA: 1.7 minutes
Step 81000 of 1000000; TPS: 8983.71; ETA: 1.7 minutes
Step 90000 of 1000000; TPS: 9085.51; ETA: 1.7 minutes
Step 99000 of 1000000; TPS: 9273.74; ETA: 1.6 minutes
Step 108000 of 1000000; TPS: 9347.44; ETA: 1.6 minutes
Step 117000 of 1000000; TPS: 9414.09; ETA: 1.6 minutes
Step 126000 of 1000000; TPS: 9462.72; ETA: 1.5 minutes
Step 135000 of 1000000; TPS: 9588.45; ETA: 1.5 minutes
Step 144000 of 1000000; TPS: 9640.22; ETA: 1.5 minutes
Step 153000 of 1000000; TPS: 9679.17; ETA: 1.5 minutes
Step 162000 of 1000000; 

In [33]:
def ellipsoid_gsd(gsd_file, new_file, ellipsoid_types, lpar, lperp):
    """Add needed information to GSD file to visualize ellipsoids.

    Saves a new GSD file with lpar and lperp values populated
    for each particle. Ovito can be used to visualize the new GSD file.

    Parameters
    ----------
    gsd_file : str
        Path to the original GSD file containing trajectory information
    new_file : str
        Path and filename of the new GSD file
    ellipsoid_types : str or list of str
        The particle types (i.e. names) of particles to be drawn
        as ellipsoids.
    lpar : float
        Value of lpar of the ellipsoids
    lperp : float
        Value of lperp of the ellipsoids

    """
    with gsd.hoomd.open(new_file, "w") as new_t:
        with gsd.hoomd.open(gsd_file) as old_t:
            for snap in old_t:
                shape_dicts_list = []
                for ptype in snap.particles.types:
                    if ptype == ellipsoid_types or ptype in ellipsoid_types:
                        shapes_dict = {
                            "type": "Ellipsoid",
                            "a": lpar,
                            "b": lperp,
                            "c": lperp,
                        }
                    else:
                        shapes_dict = {"type": "Sphere", "diameter": 0.001}
                    shape_dicts_list.append(shapes_dict)
                snap.particles.type_shapes = shape_dicts_list
                snap.validate()
                new_t.append(snap)

ellipsoid_gsd(
    gsd_file=gsd_path,
    new_file="ovito-" + GSD_FILE_PATH,
    ellipsoid_types='R',
    lpar=LPAR,lperp=LPERP
)

In [35]:
log = np.genfromtxt('log.txt', names=True)
timestep = log["flowermdbasesimulationSimulationtimestep"]
potential_energy = log["mdcomputeThermodynamicQuantitiespotential_energy"]
volume = log["mdcomputeThermodynamicQuantitiesvolume"]

plt.plot(timestep, potential_energy)
plt.title('Potential Energy vs. Time')
plt.xlabel('Timestep')
plt.ylabel('Potential Energy')
plt.savefig('graphs/potential_energy-' + time_string)
plt.close()

plt.plot(timestep, volume)
plt.title('Volume vs. Time')
plt.xlabel('Timestep')
plt.ylabel('Volume')
plt.savefig('graphs/volume-' + time_string)
plt.close()